In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def cari_akar_repo(mulai=None):
    mulai = Path.cwd() if mulai is None else Path(mulai)
    for p in [mulai, *mulai.parents]:
        if (p / "pipeline.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError("Akar repo tidak ditemukan. Buka notebook dari dalam folder repo.")

ROOT = cari_akar_repo()
RAW = ROOT / "data" / "raw"
PROFIL = ROOT / "reports" / "profil"
GAMBAR = ROOT / "reports" / "gambar"
PROFIL.mkdir(parents=True, exist_ok=True)
GAMBAR.mkdir(parents=True, exist_ok=True)
rekap_path = PROFIL / "kelengkapan_panel_sdg8_tpt.csv"

BERKAS_RAW = "bps/2026-10-17/sdg8_tpt.csv"
# angka yang sudah dilaporkan ke reviewer (K-002 dan K-003); dicocokkan, bukan dipaksakan
LAPORAN_BARIS_INDONESIA, LAPORAN_BARIS_PROVINSI, LAPORAN_SEL_KOSONG = 61, 1939, 379

PROVINSI_38 = [1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2100,
               3100, 3200, 3300, 3400, 3500, 3600, 5100, 5200, 5300,
               6100, 6200, 6300, 6400, 6500, 7100, 7200, 7300, 7400, 7500, 7600,
               8100, 8200, 9100, 9200, 9400, 9500, 9600, 9700]
assert len(PROVINSI_38) == 38
URUT_PERIODE = {"Februari": 2, "Agustus": 8, "Tahunan": 13}   # urutan kronologis dalam setahun

raw = pd.read_csv(RAW / BERKAS_RAW)
raw["nama"] = raw["wilayah"].astype(str).str.strip()
kode = pd.to_numeric(raw["wilayah_id"], errors="coerce")
assert kode.notna().all(), "wilayah_id ada yang bukan angka; lihat sel inspeksi"
raw["kode"] = np.where(kode < 100, kode * 100, kode).astype(int)   # kode BPS dua digit x 100

# --- pisahkan baris Indonesia dari baris provinsi ---
nasional = raw["nama"].str.contains("indonesia", case=False)
prov = raw[~nasional].copy()
assert set(PROVINSI_38) == set(prov["kode"]), "daftar 38 tidak sama dengan kode di data"
print("Baris mentah:", len(raw), "| Indonesia:", int(nasional.sum()), "| provinsi:", len(prov))
assert nasional.sum() == LAPORAN_BARIS_INDONESIA, "jumlah baris Indonesia beda dari K-002"
assert not prov.duplicated(["kode", "tahun", "periode"]).any(), "ada kunci kembar; tangani di tugas #13"
assert len(prov) == LAPORAN_BARIS_PROVINSI, f"baris provinsi {len(prov)} beda dari {LAPORAN_BARIS_PROVINSI}"

# --- kombinasi tahun x periode yang benar-benar terbit (rujukan: baris Indonesia) ---
kombinasi = prov[["tahun", "periode"]].drop_duplicates()
kombinasi["urut"] = kombinasi["periode"].map(URUT_PERIODE)
assert kombinasi["urut"].notna().all(), "ada nama periode di luar Februari/Agustus/Tahunan"
kombinasi = kombinasi.sort_values(["tahun", "urut"]).reset_index(drop=True)
komb_nas = set(map(tuple, raw.loc[nasional, ["tahun", "periode"]].drop_duplicates().values.tolist()))
komb_prov = set(map(tuple, kombinasi[["tahun", "periode"]].values.tolist()))
print("Kombinasi tahun x periode:", len(kombinasi),
      "| selisih terhadap baris Indonesia:", sorted(komb_nas ^ komb_prov) or "tidak ada")

# --- riwayat provinsi: sejak kapan ada di data ---
riwayat = prov.groupby("kode").agg(nama=("nama", "first"), tahun_pertama=("tahun", "min"),
                                    tahun_terakhir=("tahun", "max"), n_obs=("nilai", "count")).reset_index()
tanpa_data = sorted(set(PROVINSI_38) - set(riwayat["kode"]))
print("Provinsi tanpa satu pun data:", tanpa_data if tanpa_data else "tidak ada")

# --- grid lengkap 38 provinsi x kombinasi periode, lalu beri status tiap sel ---
grid = pd.DataFrame({"kode": PROVINSI_38}).merge(kombinasi[["tahun", "periode"]], how="cross")
grid = grid.merge(prov[["kode", "tahun", "periode", "nilai"]], on=["kode", "tahun", "periode"], how="left")
grid = grid.merge(riwayat[["kode", "nama", "tahun_pertama"]], on="kode", how="left")
grid["status"] = np.select(
    [grid["nilai"].notna(), grid["tahun_pertama"].isna() | (grid["tahun"] < grid["tahun_pertama"])],
    ["terisi", "kosong_struktural"], default="missing_aktual")

n_grid = len(grid)
n_terisi = int((grid["status"] == "terisi").sum())
n_struk = int((grid["status"] == "kosong_struktural").sum())
n_aktual = int((grid["status"] == "missing_aktual").sum())
sel_kosong_grid = n_grid - n_terisi
print(f"\nGrid {len(PROVINSI_38)} x {len(kombinasi)} = {n_grid} sel | terisi {n_terisi} | "
      f"kosong struktural {n_struk} | missing aktual {n_aktual}")
assert n_terisi == LAPORAN_BARIS_PROVINSI
assert sel_kosong_grid == LAPORAN_SEL_KOSONG, f"sel kosong {sel_kosong_grid} beda dari {LAPORAN_SEL_KOSONG}"

print("\nMissing aktual (provinsi sudah ada, periode sudah terbit, tetapi nilai tidak ada):")
print(grid.loc[grid["status"] == "missing_aktual", ["kode", "nama", "tahun", "periode", "tahun_pertama"]].to_string(index=False)
      if n_aktual else "  tidak ada")

# --- persen sel terisi per tahun (per periode, tidak dirata-rata) ---
lap = grid.groupby(["tahun", "periode"]).agg(
    sel_grid=("kode", "size"),
    terisi=("status", lambda s: int((s == "terisi").sum())),
    kosong_struktural=("status", lambda s: int((s == "kosong_struktural").sum())),
    missing_aktual=("status", lambda s: int((s == "missing_aktual").sum())),
).reset_index()
lap["sel_seharusnya_ada"] = lap["sel_grid"] - lap["kosong_struktural"]
lap["persen_terisi_dari_seharusnya"] = (lap["terisi"] / lap["sel_seharusnya_ada"] * 100).round(1)
lap["persen_terisi_dari_grid_38"] = (lap["terisi"] / lap["sel_grid"] * 100).round(1)
lap["urut"] = lap["periode"].map(URUT_PERIODE)
lap = lap.sort_values(["tahun", "urut"]).drop(columns="urut").reset_index(drop=True)

status_sel = grid[["kode", "nama", "tahun", "periode", "nilai", "status", "tahun_pertama"]].rename(columns={"kode": "wilayah_kode"})
riwayat_out = riwayat.rename(columns={"kode": "wilayah_kode"})

lap.to_csv(PROFIL / "kelengkapan_panel_per_tahun_sdg8_tpt.csv", index=False)
status_sel.to_csv(PROFIL / "status_sel_grid_sdg8_tpt.csv", index=False)
riwayat_out.to_csv(PROFIL / "riwayat_provinsi_sdg8_tpt.csv", index=False)

# --- grafik ---
fig, ax = plt.subplots(figsize=(12, 5))
for per, m in [("Tahunan", "o"), ("Februari", "s"), ("Agustus", "^")]:
    s = lap[lap["periode"] == per]
    if len(s):
        ax.plot(s["tahun"], s["persen_terisi_dari_grid_38"], marker=m, ls="--", alpha=0.5, label=f"{per}: dari 38 provinsi")
        ax.plot(s["tahun"], s["persen_terisi_dari_seharusnya"], marker=m, label=f"{per}: dari provinsi yang sudah ada")
ax.set_ylim(0, 105)
ax.set_xlabel("Tahun")
ax.set_ylabel("Sel terisi (%)")
ax.set_title("Kelengkapan panel TPT (var543): 38 provinsi x periode")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(GAMBAR / "kelengkapan_panel_sdg8_tpt.png", dpi=200)
plt.close(fig)

# --- ringkasan markdown ---
baris = ["# Kelengkapan panel TPT (var543)", "",
         f"Grid: {len(PROVINSI_38)} provinsi (susunan 2025) x {len(kombinasi)} kombinasi tahun-periode = {n_grid} sel.", "",
         f"- Terisi: {n_terisi}", f"- Kosong struktural (provinsi belum ada di data): {n_struk}",
         f"- Missing aktual (provinsi sudah ada, periode sudah terbit, nilai tidak ada): {n_aktual}", "",
         "Kosong struktural bukan missing: provinsi baru hanya berisi tahun sejak ia muncul di data (K-001). "
         "Persen terisi dihitung dua cara: dari 38 provinsi, dan dari provinsi yang sudah ada pada tahun itu.", "",
         "| Tahun | Periode | Sel seharusnya ada | Terisi | Missing aktual | % terisi (dari yang seharusnya ada) |",
         "|---|---|---|---|---|---|"]
for r in lap.itertuples():
    baris.append(f"| {r.tahun} | {r.periode} | {r.sel_seharusnya_ada} | {r.terisi} | {r.missing_aktual} | {r.persen_terisi_dari_seharusnya} |")
baris += ["", "## Sejak kapan setiap provinsi ada di data", "", "| Kode | Nama | Tahun pertama | Tahun terakhir | Observasi |", "|---|---|---|---|---|"]
for r in riwayat_out.sort_values(["tahun_pertama", "wilayah_kode"]).itertuples():
    baris.append(f"| {r.wilayah_kode} | {r.nama} | {r.tahun_pertama} | {r.tahun_terakhir} | {r.n_obs} |")
(PROFIL / "kelengkapan_panel_sdg8_tpt.md").write_text("\n".join(baris) + "\n", encoding="utf-8")

METRIK_12 = {"baris_silver": n_terisi, "sel_kosong_grid": sel_kosong_grid,
             "baris_indonesia": int(nasional.sum()), "sel_struktural": n_struk, "sel_missing_aktual": n_aktual}
print("\nMetrik untuk metrics.json:", METRIK_12)
print("Provinsi menurut tahun pertama muncul (selain tahun awal seri):")
print(riwayat_out[riwayat_out["tahun_pertama"] > raw["tahun"].min()].sort_values("tahun_pertama")[["wilayah_kode", "nama", "tahun_pertama"]].to_string(index=False))

Baris mentah: 2000 | Indonesia: 61 | provinsi: 1939
Kombinasi tahun x periode: 61 | selisih terhadap baris Indonesia: tidak ada
Provinsi tanpa satu pun data: tidak ada

Grid 38 x 61 = 2318 sel | terisi 1939 | kosong struktural 378 | missing aktual 1

Missing aktual (provinsi sudah ada, periode sudah terbit, tetapi nilai tidak ada):
 kode   nama  tahun periode  tahun_pertama
 8100 MALUKU   2000 Tahunan           1986

Metrik untuk metrics.json: {'baris_silver': 1939, 'sel_kosong_grid': 379, 'baris_indonesia': 61, 'sel_struktural': 378, 'sel_missing_aktual': 1}
Provinsi menurut tahun pertama muncul (selain tahun awal seri):
 wilayah_kode                 nama  tahun_pertama
         1900 KEP. BANGKA BELITUNG           2001
         3600               BANTEN           2001
         7500            GORONTALO           2001
         8200         MALUKU UTARA           2001
         2100            KEP. RIAU           2006
         7600       SULAWESI BARAT           2006
         9100       

In [2]:
print(raw[raw["kode"].isin([9300, 9700])][["wilayah_id", "wilayah", "tahun", "periode"]]
      .groupby(["wilayah_id", "wilayah"]).agg(tahun_min=("tahun", "min"), tahun_maks=("tahun", "max"), n=("tahun", "size")))
print(sorted(set(raw.loc[~nasional, "kode"])))
print(len(set(raw.loc[~nasional, "kode"])))

                             tahun_min  tahun_maks  n
wilayah_id wilayah                                   
9700       PAPUA PEGUNUNGAN       2024        2026  5
[1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2100, 3100, 3200, 3300, 3400, 3500, 3600, 5100, 5200, 5300, 6100, 6200, 6300, 6400, 6500, 7100, 7200, 7300, 7400, 7500, 7600, 8100, 8200, 9100, 9200, 9400, 9500, 9600, 9700]
38


In [3]:
print("Total raw:", len(raw), "| Nasional:", int(nasional.sum()), "| Non-nasional:", int((~nasional).sum()))
print("Nilai NaN di prov:", int(prov["nilai"].isna().sum()))
print("Baris nasional:", raw.loc[nasional, ["wilayah_id", "kode", "wilayah", "tahun", "periode"]].drop_duplicates("wilayah").to_string(index=False))

Total raw: 2000 | Nasional: 61 | Non-nasional: 1939
Nilai NaN di prov: 0
Baris nasional:  wilayah_id  kode   wilayah  tahun periode
       9999  9999 INDONESIA   2000 Tahunan
